# Práctica 3.1 — Entrenar y servir un bot sencillo (Nebula Arena)
Ejecuta este notebook en **Amazon SageMaker Studio Lab** o **Google Colab** (con entorno GPU T4 activado) para comparar el entrenamiento en `cpu` vs `cuda` y medir la latencia de inferencia.

In [ ]:
!pip install -q gymnasium stable-baselines3 torch

In [ ]:
import gymnasium as gym, time, math
import numpy as np
import torch
from stable_baselines3 import PPO

env = gym.make("CartPole-v1")

# 1. Entrenamiento en CPU
modelo_cpu = PPO("MlpPolicy", env, verbose=0, device="cpu")
inicio = time.time()
modelo_cpu.learn(total_timesteps=100_000)
t_cpu = time.time() - inicio
print(f"Entrenamiento CPU: {t_cpu:.1f} s")
modelo_cpu.save("bot_nebula")

# 2. Entrenamiento en GPU (si está disponible)
if torch.cuda.is_available():
    modelo_gpu = PPO("MlpPolicy", env, verbose=0, device="cuda")
    inicio = time.time()
    modelo_gpu.learn(total_timesteps=100_000)
    t_gpu = time.time() - inicio
    print(f"Entrenamiento GPU (CUDA): {t_gpu:.1f} s")
else:
    print("GPU CUDA no activa en este entorno.")

In [ ]:
# 3. Medición de latencia de inferencia en CPU
obs, _ = env.reset()
t = time.perf_counter()
for _ in range(10_000):
    accion, _ = modelo_cpu.predict(obs, deterministic=True)
lat_ms = (time.perf_counter() - t) / 10_000 * 1000
dec_s = 1000 / lat_ms
vcpu_necesarias = math.ceil((2000 * 10 / dec_s) * 1.30)
print(f"Latencia media: {lat_ms:.3f} ms -> {dec_s:,.0f} decisiones/s por núcleo")
print(f"vCPU necesarias para 2,000 bots a 10 Hz (+30% margen): {vcpu_necesarias} vCPU")